# Generador v3, paso 2: QLoRA con pasajes de la app y causas del pasaje (Colab T4)

Mismo modelo y mismos hiperparámetros que el v1 y el v2 (Qwen2.5-3B-Instruct, 4 bits, LoRA r = 16, lr 2e-4,
2 épocas, lote efectivo 16). Cambian los datos (`sft_train_v3.jsonl`):
- Los pasajes son los de la app: para un examen alto o bajo, el fragmento que explica ese resultado.
- Las respuestas son del profesor v3, que toma las causas del pasaje. Se filtran con los criterios del v2 y,
  además, sin ninguna causa que no esté en el pasaje (`verificacion.causas_sin_respaldo`).

Antes de abrirlo, `sft_train_v3.jsonl` tiene que estar en GitHub (`git push`), porque este notebook clona el repositorio.

Tiempo aproximado: entrenar unos 40 minutos y exportar a GGUF unos 20 minutos. Las respuestas del test se generan
en el computador local con los GGUF del v1, el v2 y el v3, sobre los mismos casos (`casos_v3.jsonl`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
ADAPTADOR = f'{DRIVE}/qwen2.5-3b-instruct-lora-v3'
GGUF = f'{DRIVE}/letrademedico-qwen2.5-3b-v3-q4_k_m.gguf'
!mkdir -p {DRIVE}/checkpoints-v3

In [ ]:
%cd /content
!test -d Proyecto_LaLetraDeMedico || git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd /content/Proyecto_LaLetraDeMedico
!git pull -q && git log --oneline -1
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets
!wc -l data/generador/sft_train_v3.jsonl data/generador/sft_val_v3.jsonl

## 1. Entrenar el v3 (se reanuda desde el último checkpoint si Colab se desconectó)

In [ ]:
%cd /content/Proyecto_LaLetraDeMedico
!python training/generador/ajustar_qlora.py --version v3 --checkpoints {DRIVE}/checkpoints-v3 --salida {ADAPTADOR}
!cat {ADAPTADOR}/entrenamiento.json | head -c 600

## 2. Exportar a GGUF Q4_K_M

Fusiona el LoRA con el modelo base en float16 (en CPU) y cuantiza a 4 bits para llama.cpp.

In [ ]:
# Colab trae torchao 0.10, que la versión actual de peft rechaza; no se usa, así que se quita
!pip uninstall -y -q torchao
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', dtype=torch.float16, device_map='cpu')
fusionado = PeftModel.from_pretrained(base, ADAPTADOR).merge_and_unload()
fusionado.save_pretrained('/content/fusionado-v3', safe_serialization=True)
AutoTokenizer.from_pretrained(ADAPTADOR).save_pretrained('/content/fusionado-v3')
del base, fusionado

In [ ]:
%cd /content
!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py /content/fusionado-v3 --outfile /content/ldm-v3-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize /content/ldm-v3-f16.gguf "{GGUF}" Q4_K_M
!ls -lh "{GGUF}"

## 3. Descargar el adaptador (pequeño)

In [ ]:
!rm -rf /content/v3 && mkdir -p /content/v3/models/generador
!cp -r {ADAPTADOR} /content/v3/models/generador/
%cd /content/v3
!zip -qr /content/generador_v3.zip models && ls -lh /content/generador_v3.zip
from google.colab import files
files.download('/content/generador_v3.zip')

## 4. En el computador local

1. Descomprime `generador_v3.zip` en la raíz del proyecto (deja `models/generador/qwen2.5-3b-instruct-lora-v3/`).
2. Descarga desde Google Drive (`MyDrive/letrademedico/`) el archivo `letrademedico-qwen2.5-3b-v3-q4_k_m.gguf`,
   de unos 2 GB, y cópialo a `models/generador/`.
3. Avísale a Claude: genera las respuestas del test con los tres GGUF (`generar_respuestas_gguf.py`) y compara
   con `python evaluacion/evaluar_generador.py --version v3`.